# 03 — Semantica Decision Intelligence on real chart-review runs

**Question:** what becomes possible after verified Decision Episodes from many runs are
stored in one ContextGraph?

This notebook demonstrates the Semantica capabilities that are actually relevant to
ACR:

- ContextGraph as the storage/query substrate;
- native Decision recording and scoped insights;
- similar-decision retrieval and divergent Decision Points;
- explicit causal-chain traversal and candidate impact analysis;
- Policy versioning and direct-binding impact queues;
- native provenance lineage, integrity checks, and optional PROV export.

It does **not** use GraphRAG, Semantica's general rule Reasoner, Ontology Hub, generic
`KGVisualizer`, or the general graph export framework. Explorer is the interactive UI;
the notebook focuses on reproducible analytical queries.


## 1. Load a multi-run ContextGraph

“Decision graph” below means the decision-centered subgraph inside ContextGraph, not a
second ACR graph. Decision nodes hold a de-identified comparable signature. Detailed
state, testimony, actions, observations, and field provenance live in Semantica's
run-local ProvenanceManager and are linked from the projection.


In [1]:
from collections import Counter, defaultdict
from pathlib import Path
import json
import os
import shutil

from IPython.display import Markdown, display
from acr.mvp.ledger import SemanticaLedger

START_DIR = Path.cwd().resolve()
ROOT = START_DIR if (START_DIR / "pyproject.toml").is_file() else START_DIR.parent
assert (ROOT / "pyproject.toml").is_file(), "Start Jupyter from the repo or notebooks/"
seed = ROOT / "runs/policy-experiment-20260827/experiment-ledger.json"
generated = ROOT / "runs/postdoc-study/ledger.json"
default_ledger = seed if seed.is_file() else generated
LEDGER_PATH = Path(os.environ.get("ACR_INTELLIGENCE_LEDGER", default_ledger))
assert LEDGER_PATH.is_file(), "Run Notebook 1 first or set ACR_INTELLIGENCE_LEDGER"
ledger = SemanticaLedger(LEDGER_PATH)

def one_line(value, limit=105):
    text = " ".join(str("" if value is None else value).split())
    return text if len(text) <= limit else text[: limit - 1] + "…"

def display_path(value):
    path = Path(value).resolve()
    try:
        return str(path.relative_to(ROOT))
    except ValueError:
        return str(path)

def markdown_table(rows, columns):
    def safe(value):
        return one_line(value, 130).replace("|", "/")
    return "\n".join([
        "| " + " | ".join(label for _, label in columns) + " |",
        "|" + "|".join("---" for _ in columns) + "|",
        *("| " + " | ".join(safe(row.get(key, "")) for key, _ in columns) + " |"
          for row in rows),
    ])

stats = ledger.stats()
display({"ledger": display_path(LEDGER_PATH), **stats})


{'ledger': 'runs/policy-experiment-20260827/experiment-ledger.json',
 'analyses': 8,
 'episodes': 69,
 'selections': 2,
 'causal_assertions': 38,
 'semantica': {'node_count': 1165,
  'edge_count': 2299,
  'node_types': {'AnalysisArtifact': 8,
   'ReActCycle': 212,
   'StateSnapshot': 424,
   'DecisionTestimony': 67,
   'EvidenceRef': 161,
   'RuntimeNoteFinding': 28,
   'Rule': 55,
   'Submission': 8,
   'GateVerdict': 8,
   'RunResult': 8,
   'decision': 69,
   'entity': 35,
   'category': 4,
   'decision_maker': 1,
   'CausalAssertion': 38,
   'ProjectionManifest': 8,
   'PolicyBundle': 1,
   'Policy': 22,
   'DiscriminatingFact': 6,
   'AnalysisSelection': 2},
  'edge_types': {'STARTED_WITH': 212,
   'ENDED_WITH': 212,
   'OCCURRED_IN': 67,
   'CITES': 438,
   'CONTAINS_FINDING': 28,
   'TESTIFIES_TO': 95,
   'CONTAINS_EVIDENCE': 14,
   'CONTAINS_EXECUTION': 16,
   'EVALUATED_AS': 8,
   'PRODUCED': 8,
   'involves': 430,
   'belongs_to': 69,
   'made_by': 69,
   'CONTAINS_EPISODE': 

## 2. See what is in the ContextGraph

The important distinction is between reusable comparison nodes and audit detail:

- `decision`, `Policy`, and typed causal edges support cross-run queries.
- `ReActCycle`, `StateSnapshot`, `DecisionTestimony`, evidence/rule pointers, and
  `CausalAssertion` preserve the graph path back to execution evidence.
- the full payload remains in the content-addressed artifact and provenance database,
  avoiding oversized or patient-identifying Decision signatures.


In [2]:
graph_stats = stats["semantica"]
node_rows = [
    {"type": key, "count": value}
    for key, value in sorted(
        graph_stats["node_types"].items(), key=lambda item: (-item[1], item[0])
    )
]
edge_rows = [
    {"type": key, "count": value}
    for key, value in sorted(
        graph_stats["edge_types"].items(), key=lambda item: (-item[1], item[0])
    )
]
display(Markdown("### Node types\n" + markdown_table(node_rows, [
    ("type", "Node type"), ("count", "Count")
])))
display(Markdown("### Relationship types\n" + markdown_table(edge_rows, [
    ("type", "Edge type"), ("count", "Count")
])))


### Node types
| Node type | Count |
|---|---|
| StateSnapshot | 424 |
| ReActCycle | 212 |
| EvidenceRef | 161 |
| decision | 69 |
| DecisionTestimony | 67 |
| Rule | 55 |
| CausalAssertion | 38 |
| entity | 35 |
| RuntimeNoteFinding | 28 |
| Policy | 22 |
| AnalysisArtifact | 8 |
| GateVerdict | 8 |
| ProjectionManifest | 8 |
| RunResult | 8 |
| Submission | 8 |
| DiscriminatingFact | 6 |
| category | 4 |
| AnalysisSelection | 2 |
| PolicyBundle | 1 |
| decision_maker | 1 |

### Relationship types
| Edge type | Count |
|---|---|
| CITES | 438 |
| involves | 430 |
| ENDED_WITH | 212 |
| STARTED_WITH | 212 |
| DERIVED_FROM | 204 |
| TESTIFIES_TO | 95 |
| APPLIED_POLICY | 75 |
| CONTAINS_EPISODE | 69 |
| belongs_to | 69 |
| made_by | 69 |
| OCCURRED_IN | 67 |
| CHECKED | 62 |
| SUPPORTED_BY | 41 |
| ASSERTS_SOURCE | 38 |
| ASSERTS_TARGET | 38 |
| INFLUENCED | 38 |
| CONTAINS_FINDING | 28 |
| GOVERNED_BY_POLICY_BUNDLE | 22 |
| COMPOSED_OF | 21 |
| CONTAINS_EXECUTION | 16 |
| CONTAINS_EVIDENCE | 14 |
| USES | 14 |
| EVALUATED_AS | 8 |
| PRODUCED | 8 |
| SUBMITTED_AS | 8 |
| SELECTS | 2 |
| VERSION_OF | 1 |

## 3. Find the same Decision Point with different outcomes

A final-answer mismatch says only that two runs differ. A divergent Decision Point is
more actionable: same atomic function, subject, and pre-decision situation; different
committed outcome.

We first restrict the cohort to one reconstruction per `task_only` run. Semantica
supplies native similar-decision candidates; ACR applies chart-review identity and
cohort guards. An ungrounded divergence is a **guideline-question candidate**, not an
automatic verdict that either model was clinically wrong.


In [3]:
analyses_by_run = defaultdict(set)
for node in ledger.graph.find_nodes(node_type="decision"):
    meta = node.get("metadata") or {}
    if meta.get("run_id") and meta.get("analysis_id"):
        analyses_by_run[str(meta["run_id"])].add(str(meta["analysis_id"]))

task_only_cohort = []
for candidate_run, analysis_ids in sorted(analyses_by_run.items()):
    matching = [
        analysis_id for analysis_id in sorted(analysis_ids)
        if ledger.load_analysis_artifact(candidate_run, analysis_id).get("task_arm")
        == "task_only"
    ]
    if matching:
        selected = ledger.selected_analysis(candidate_run)
        task_only_cohort.append({
            "run_id": candidate_run,
            "analysis_id": selected if selected in matching else matching[0],
        })

divergence_report = ledger.find_divergent_decision_points(
    task_only_cohort, min_similarity=0.68
) if len(task_only_cohort) >= 2 else {"divergences": []}
exact_divergences = [
    row for row in divergence_report.get("divergences", [])
    if row.get("same_situation_signature")
]
if exact_divergences:
    divergence = exact_divergences[0]
    scenario_texts = {
        (ledger.graph.find_node(str(member["decision_id"])).get("metadata") or {})
        .get("scenario")
        for member in divergence["members"]
    }
    exact_scenario = next(iter(scenario_texts)) if len(scenario_texts) == 1 else (
        "Scenario strings did not agree; inspect the signature collision."
    )
    rows = [{
        "model": member.get("review_model"),
        "run": str(member.get("run_id"))[:24] + "…",
        "outcome": member.get("outcome"),
        "basis": ", ".join(member.get("basis_sources") or []),
        "policy": ", ".join(member.get("policy_groundings") or []) or "none",
    } for member in divergence["members"]]
    display(Markdown(
        f"**Decision Point:** `{divergence['decision_function']}/"
        f"{divergence['decision_subject']}`  \n"
        f"**Exact scenario:** `{exact_scenario}`  \n"
        f"**Routing:** `{divergence['grounding_status']}`\n\n" +
        markdown_table(rows, [
            ("model", "Review model"), ("outcome", "Outcome"),
            ("basis", "Claimed basis"), ("policy", "Policy grounding"),
            ("run", "Run"),
        ])
    ))
else:
    divergence = None
    display(Markdown(
        "**No exact divergent Decision Point was found in this cohort.** That is a "
        "valid experimental result, not a query failure. Add repetitions/models in "
        "Notebook 1 before concluding the guideline is stable."
    ))


**Decision Point:** `standing/evidence_item`  
**Exact scenario:** `predecision decision_function=standing decision_subject=evidence_item candidate_shape=multiple conflict_shape=competing_candidates surfaced_shape=multiple read_shape=multiple finding_shape=none uncertainty_shape=single fields=none evidence_roles=none standings=none`  
**Routing:** `UNGROUNDED_OUTCOME_DIVERGENCE`

| Review model | Outcome | Claimed basis | Policy grounding | Run |
|---|---|---|---|---|
| openai/gpt-5.6-luna | MERELY_MENTIONS | chart, own_knowledge | none | 20260827T101823421486Z_S… |
| openai/gpt-5.6-terra | CAN_ESTABLISH | chart, own_knowledge | none | 20260827T102506244464Z_S… |

## 4. Retrieve similar decisions as comparison candidates

Semantica's native `ContextGraph.find_similar_decisions` returns nearby Decision
scenarios. ACR then excludes the query run and requires the intended atomic subject.
Similarity proposes where a human should compare; it does not establish equivalence,
precedent, or clinical correctness.


In [4]:
if divergence:
    query_episode_id = divergence["members"][0]["episode_id"]
else:
    query_node = next(iter(ledger.graph.find_nodes(node_type="decision")))
    query_episode_id = (query_node.get("metadata") or {})["acr_episode_id"]
similar_report = ledger.similar_candidates(
    query_episode_id, max_results=8, min_similarity=0.45
)
similar_rows = []
for candidate in similar_report["candidates"]:
    decision = candidate["decision"]
    meta = decision.get("metadata") or {}
    similar_rows.append({
        "similarity": f"{float(candidate.get('similarity') or 0):.2f}",
        "same_signature": candidate.get("same_situation_signature"),
        "category": decision.get("category"),
        "subject": meta.get("decision_subject"),
        "outcome": decision.get("outcome"),
        "run": str(meta.get("run_id"))[:24] + "…",
    })
assert similar_report["retrieval_engine"] == (
    "semantica.ContextGraph.find_similar_decisions"
)
display(Markdown(markdown_table(similar_rows, [
    ("similarity", "Similarity"), ("same_signature", "Exact signature"),
    ("category", "Function"), ("subject", "Subject"),
    ("outcome", "Outcome"), ("run", "Run"),
]) if similar_rows else "No cross-run candidate met the threshold."))


| Similarity | Exact signature | Function | Subject | Outcome | Run |
|---|---|---|---|---|---|
| 0.72 | True | standing | evidence_item | MERELY_MENTIONS | 20260827T105131502195Z_S… |
| 0.72 | True | standing | evidence_item | CAN_ESTABLISH | 20260827T102506244464Z_S… |
| 0.72 | True | standing | evidence_item | MERELY_MENTIONS | 20260827T105131502195Z_S… |
| 0.67 | False | standing | evidence_item | CAN_ESTABLISH | 20260827T105508151402Z_S… |
| 0.67 | False | standing | evidence_item | CAN_ESTABLISH | 20260827T102506309233Z_S… |
| 0.62 | False | standing | evidence_item | CAN_ESTABLISH | 20260827T102506274392Z_S… |
| 0.51 | False | standing | evidence_item | CAN_ESTABLISH | 20260827T105131502195Z_S… |
| 0.51 | False | standing | evidence_item | CAN_ESTABLISH | 20260827T105131502195Z_S… |

## 5. Traverse an evidenced causal chain

ACR writes Semantica `CAUSED`, `INFLUENCED`, or `PRECEDENT_FOR` only when a
`CausalAssertion` carries supporting runtime references. The query intentionally
excludes Semantica's heuristic lowercase `influences` edges and mere temporal
adjacency.

We choose the latest episode with an evidenced incoming chain in an explicitly selected
policy-guided analysis and ask what led to it.
This is the data behind the human “why did the next step happen?” path; it is not the
stock generic graph visualization.


In [5]:
selections = ledger.graph.find_nodes(node_type="AnalysisSelection")
assert selections, "Causal review requires an explicitly selected analysis"
selection_candidates = []
for selection in selections:
    meta = selection.get("metadata") or {}
    candidate_artifact = ledger.load_analysis_artifact(
        str(meta["run_id"]), str(meta["analysis_id"])
    )
    selection_candidates.append((
        candidate_artifact.get("task_arm") == "policy_bundle",
        str(meta["run_id"]),
        str(meta["analysis_id"]),
        meta,
    ))
selected_meta = max(selection_candidates, key=lambda row: row[:3])[3]
causal_run = str(selected_meta["run_id"])
causal_analysis = str(selected_meta["analysis_id"])
scoped_decisions = [
    node for node in ledger.graph.find_nodes(node_type="decision")
    if (node.get("metadata") or {}).get("run_id") == causal_run
    and (node.get("metadata") or {}).get("analysis_id") == causal_analysis
]
ordered_decisions = sorted(
    scoped_decisions,
    key=lambda node: int((node.get("metadata") or {}).get("source_seq_start") or 0),
)
final_node = None
causal = None
for candidate_node in reversed(ordered_decisions):
    candidate_episode = str(
        (candidate_node.get("metadata") or {})["acr_episode_id"]
    )
    candidate_trace = ledger.causal_trace(candidate_episode, max_steps=12)
    if candidate_trace["chains"]:
        final_node, causal = candidate_node, candidate_trace
        break
assert final_node is not None and causal is not None, (
    "The selected analysis has no evidenced causal chain"
)
final_episode = str((final_node.get("metadata") or {})["acr_episode_id"])
hop_rows = []
graph_edge_rows = [
    edge.to_dict() if hasattr(edge, "to_dict") else dict(edge)
    for edge in ledger.graph.edges
]
for chain_index, chain in enumerate(causal["chains"], 1):
    for hop_index, hop in enumerate(chain.get("hops") or [], 1):
        source = ledger.graph.find_node(str(hop["from"])) or {}
        target = ledger.graph.find_node(str(hop["to"])) or {}
        assertion_node = next((
            node for node in ledger.graph.find_nodes(node_type="CausalAssertion")
            if (node.get("metadata") or {}).get("assertion_id")
            == hop.get("assertion_id")
        ), None)
        support_count = sum(
            edge.get("type") == "SUPPORTED_BY"
            and assertion_node is not None
            and edge.get("source_id") == assertion_node.get("id")
            for edge in graph_edge_rows
        )
        hop_rows.append({
            "chain": chain_index,
            "hop": hop_index,
            "from": (source.get("metadata") or {}).get("category"),
            "type": hop.get("type"),
            "to": (target.get("metadata") or {}).get("category"),
            "assertion": hop.get("assertion_id"),
            "provenance": hop.get("assertion_provenance"),
            "support": support_count,
        })
assert hop_rows and all(row["assertion"] for row in hop_rows)
display(Markdown(markdown_table(hop_rows, [
    ("chain", "Chain"), ("hop", "Hop"), ("from", "From"),
    ("type", "Relationship"), ("to", "To"),
    ("assertion", "Causal assertion"), ("provenance", "Provenance"),
    ("support", "Support nodes"),
])))


| Chain | Hop | From | Relationship | To | Causal assertion | Provenance | Support nodes |
|---|---|---|---|---|---|---|---|
| 1 | 1 | where_to_look | INFLUENCED | where_to_look | analysis-96d8ce57c0016c307499:runtime-dependency:1 | DETERMINISTIC_DERIVED_FROM_RUNTIME_REFERENCE | 3 |
| 1 | 2 | where_to_look | INFLUENCED | standing | analysis-96d8ce57c0016c307499:runtime-dependency:2 | DETERMINISTIC_DERIVED_FROM_RUNTIME_REFERENCE | 1 |
| 1 | 3 | standing | INFLUENCED | which_wins | analysis-96d8ce57c0016c307499:runtime-dependency:5 | DETERMINISTIC_DERIVED_FROM_RUNTIME_REFERENCE | 1 |
| 2 | 1 | where_to_look | INFLUENCED | where_to_look | analysis-96d8ce57c0016c307499:runtime-dependency:1 | DETERMINISTIC_DERIVED_FROM_RUNTIME_REFERENCE | 3 |
| 2 | 2 | where_to_look | INFLUENCED | standing | analysis-96d8ce57c0016c307499:runtime-dependency:4 | DETERMINISTIC_DERIVED_FROM_RUNTIME_REFERENCE | 1 |
| 2 | 3 | standing | INFLUENCED | which_wins | analysis-96d8ce57c0016c307499:runtime-dependency:7 | DETERMINISTIC_DERIVED_FROM_RUNTIME_REFERENCE | 1 |
| 3 | 1 | where_to_look | INFLUENCED | where_to_look | analysis-96d8ce57c0016c307499:runtime-dependency:1 | DETERMINISTIC_DERIVED_FROM_RUNTIME_REFERENCE | 3 |
| 3 | 2 | where_to_look | INFLUENCED | standing | analysis-96d8ce57c0016c307499:runtime-dependency:3 | DETERMINISTIC_DERIVED_FROM_RUNTIME_REFERENCE | 1 |
| 3 | 3 | standing | INFLUENCED | which_wins | analysis-96d8ce57c0016c307499:runtime-dependency:6 | DETERMINISTIC_DERIVED_FROM_RUNTIME_REFERENCE | 1 |

## 6. Ask for impact candidates — and keep the authority boundary visible

Semantica can rank decisions that may have been influenced by a selected source. This
native analysis includes graph, entity, category, and temporal signals; it is broader
than the evidenced audit chain above. ACR therefore returns it as `CANDIDATE_ONLY`: it
identifies decisions worth re-auditing, not a counterfactual claim that changing the
source would change the final answer.


In [6]:
first_hop = causal["chains"][0]["hops"][0]
source_node = ledger.graph.find_node(str(first_hop["from"]))
source_episode = str((source_node.get("metadata") or {})["acr_episode_id"])
impact = ledger.impact_candidates(source_episode)
native_impact = impact["candidates"]
influence_rows = []
for candidate in (native_impact.get("influence_scores") or [])[:8]:
    influence_rows.append({
        "relation": "direct" if candidate.get("is_direct") else "indirect",
        "score": f"{float(candidate.get('score') or 0):.2f}",
        "category": candidate.get("category"),
        "outcome": candidate.get("outcome"),
        "run": str(((candidate.get("decision") or {}).get("metadata") or {})
                   .get("run_id"))[:24] + "…",
    })
display(Markdown(
    f"**Authority:** `{impact['authority']}`; engine: "
    f"`{impact['retrieval_engine']}`; total candidates: "
    f"**{native_impact.get('total_influenced', 0)}**.\n\n" +
    (markdown_table(influence_rows, [
        ("relation", "Candidate relation"), ("score", "Score"),
        ("category", "Function"), ("outcome", "Outcome"), ("run", "Run"),
    ]) if influence_rows else "No candidate met Semantica's native thresholds.") +
    "\n\nCannot establish: " + ", ".join(impact["cannot_establish"])
))
assert impact["authority"] == "CANDIDATE_ONLY"


**Authority:** `CANDIDATE_ONLY`; engine: `semantica.ContextGraph.analyze_decision_impact`; total candidates: **63**.

| Candidate relation | Score | Function | Outcome | Run |
|---|---|---|---|---|
| direct | 1.00 | where_to_look | ACTION:SEARCH | 20260827T101823421486Z_S… |
| direct | 1.00 | where_to_look | ACTION:SEARCH | 20260827T101823421486Z_S… |
| direct | 1.00 | where_to_look | ACTION:SEARCH | 20260827T105508151402Z_S… |
| direct | 1.00 | where_to_look | ACTION:SEARCH | 20260827T105131502195Z_S… |
| direct | 1.00 | where_to_look | ACTION:SEARCH | 20260827T102506309233Z_S… |
| direct | 1.00 | where_to_look | ACTION:SEARCH | 20260827T102506309233Z_S… |
| direct | 1.00 | where_to_look | ACTION:SEARCH | 20260827T102506274392Z_S… |
| direct | 1.00 | where_to_look | ACTION:SEARCH | 20260827T102506274392Z_S… |

Cannot establish: causation, counterfactual impact

## 7. Revise one Policy and retrieve the exact historical re-audit queue

This cell works on a scratch copy of the ledger. It chooses a directly applied Policy,
appends a new content-addressed version, and asks Semantica's `PolicyEngine` which
historical decisions were bound to the old version.

The output is precise about **historical direct bindings**. It cannot tell us whether
the revised policy would change those decisions or their final answers; that requires
paired reruns with a new Task Presentation.


In [7]:
raw_edges = [edge.to_dict() if hasattr(edge, "to_dict") else dict(edge)
             for edge in ledger.graph.edges]
applied_counts = Counter(
    str(edge["target_id"]) for edge in raw_edges if edge.get("type") == "APPLIED_POLICY"
)
assert applied_counts, "This cohort has no directly applied Policy"
old_policy_node_id, binding_count = applied_counts.most_common(1)[0]
old_policy = ledger.graph.find_node(old_policy_node_id)
old_meta = old_policy.get("metadata") or {}
policy_id = str(old_meta["policy_id"])
from_version = str(old_meta["version"])

scratch_root = ROOT / "runs/postdoc-notebook-output"
scratch_root.mkdir(parents=True, exist_ok=True)
scratch_path = scratch_root / "03_policy_sandbox.json"
shutil.copy2(LEDGER_PATH, scratch_path)
scratch = SemanticaLedger(scratch_path)
revised_rules = json.loads(json.dumps(old_meta["rules"]))
revised_rules["tutorial_change"] = (
    "Clarify this clause; demonstration only, not an approved clinical revision."
)
revision = scratch.register_policy_revision(
    policy_id,
    from_version=from_version,
    rules=revised_rules,
    change_reason="Postdoc notebook demonstration; not an approved guideline change.",
)
affected = scratch.affected_by_policy_change(
    policy_id,
    from_version=from_version,
    to_version=revision["version"],
)
affected_rows = [{
    "run": str(row.get("run_id"))[:24] + "…",
    "analysis": row.get("analysis_id"),
    "decisions": row.get("affected_decision_count"),
    "functions": ", ".join(row.get("decision_functions") or []),
} for row in affected["affected_cases"]]
display(Markdown(
    f"Policy `{policy_id}` had **{binding_count}** direct bindings in the source "
    f"ledger. New version: `{revision['version']}`.\n\n" +
    markdown_table(affected_rows, [
        ("run", "Run"), ("analysis", "Analysis"),
        ("decisions", "Affected decisions"), ("functions", "Functions"),
    ]) +
    "\n\n**Authority:** `" + affected["authority"] + "`; cannot establish: " +
    ", ".join(affected["cannot_establish"])
))


Policy `STORE.390.date_of_initial_diagnosis.conflict.physician_statement_predating_tissue` had **16** direct bindings in the source ledger. New version: `content-70562c9abea9`.

| Run | Analysis | Affected decisions | Functions |
|---|---|---|---|
| 20260827T105131502195Z_S… | analysis-5f341749e241e064d87d | 5 | standing, where_to_look, which_wins |
| 20260827T105131502195Z_S… | analysis-96d8ce57c0016c307499 | 5 | standing, where_to_look, which_wins |
| 20260827T105508151402Z_S… | analysis-f0c23d111afbe25e6e27 | 6 | standing, what_to_answer, where_to_look, which_wins |

**Authority:** `RE_AUDIT_CANDIDATES_ONLY`; cannot establish: changed clinical answer, automatic non-compliance

## 8. Inspect native provenance for one Decision

The Decision node is a stable analytical index. Semantica ProvenanceManager retains
who/what produced it, the reconstruction activity, parent analysis artifact, used
cycles/testimony/basis references, checksums, and the complete episode payload with
field-level authority.

`verify_chain()` checks storage integrity and parent links. It does not validate that a
medical interpretation is correct.


In [8]:
provenance = ledger.provenance_manager(causal_run, causal_analysis)
record = provenance.get_provenance(str(final_node["id"]))
assert record, "The selected Decision has no native provenance record"
integrity = provenance.verify_chain()
provenance_summary = {
    "decision_id": record["entity_id"],
    "entity_type": record["entity_type"],
    "reconstructor_agent": record["agent_id"],
    "role": record["role"],
    "activity_id": record["activity_id"],
    "parent_analysis_artifact": record["parent_entity_id"],
    "used_entities": record["used_entities"],
    "checksum_prefix": str(record["checksum"])[:16],
    "episode_field_provenance": (record.get("metadata") or {})
        .get("episode", {}).get("field_provenance"),
    "integrity": integrity,
}
display(provenance_summary)
assert integrity["valid"] is True


{'decision_id': '209ca8d8-0fd4-45c5-af84-561ada0a397f',
 'entity_type': 'decision',
 'reconstructor_agent': 'openrouter/openai/gpt-5.6-luna',
 'role': 'decision_reconstructor',
 'activity_id': 'acr:reconstruction-activity:6385f0351d4355962a958d80',
 'parent_analysis_artifact': 'acr:analysis-artifact:47dfa6315be360b996c9bd98',
 'used_entities': ['acr:react-cycle:f37c8210714842fda50a237b',
  'acr:react-cycle:9d57a80e8baa9c6f196dd742',
  'acr:decision-testimony:984819ac4cd4401571d58b67',
  'acr:basis-reference:57fc5db3f3f28e1ec7f03c4b',
  'acr:basis-reference:e354f50140a3517d25bab88a',
  'acr:basis-reference:4b35171da957ed3ae85da86c',
  'acr:basis-reference:a05d90a861b1f171bf0dda30',
  'acr:basis-reference:9e146cc7ee14245fdf73dcbf',
  'acr:basis-reference:456d525352f8b3c7ce1f3788',
  'acr:basis-reference:d4b9e4cdfca2d9485f76b70b',
  'acr:basis-reference:903c0159da879224e7fb9fe9',
  'acr:basis-reference:c1f4a63ccfaacbf740304b2e',
  'acr:basis-reference:a45cb40262e9b6be680ca417',
  'acr:bas

## 9. Optional capability: export provenance as W3C PROV Turtle

ACR does not currently use Semantica's general graph Export module. The native
ProvenanceManager can nevertheless serialize the lineage it already stores. This is a
useful integration experiment for an external audit archive, so we demonstrate it
without treating it as part of the maintained pipeline.


In [9]:
turtle = provenance.export_prov(format="turtle")
turtle_path = ROOT / "runs/postdoc-notebook-output/selected-run-provenance.ttl"
turtle_path.write_text(turtle)
display({
    "experimental_export": display_path(turtle_path),
    "bytes": len(turtle.encode("utf-8")),
    "first_lines": turtle.splitlines()[:8],
    "production_status": "EXPLORATORY_NOT_MAINTAINED_WORKFLOW",
})


{'experimental_export': 'runs/postdoc-notebook-output/selected-run-provenance.ttl',
 'bytes': 741868,
 'first_lines': ['@prefix ex: <https://semantica.dev/ns#> .',
  '@prefix prov: <http://www.w3.org/ns/prov#> .',
  '@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .',
  '',
  '<https://semantica.dev/ns#bundle_acr:decision-bundle:4df64ab34ee77888c3f205af> a prov:Bundle ;',
  '    prov:hadMember ex:13c25ffb-1647-418b-8eb1-3433a8a147f7,',
  '        ex:209ca8d8-0fd4-45c5-af84-561ada0a397f,',
  '        ex:414a0149-1dd7-43f7-8803-8de7c1e95b53,'],
 'production_status': 'EXPLORATORY_NOT_MAINTAINED_WORKFLOW'}

## 10. Run-scoped Decision insights

Semantica's insight summary is recomputed on one explicitly selected run/analysis so
repeated reconstructions do not inflate counts. Here `confidence` means reconstruction
stability, not clinical correctness.


In [10]:
insights = ledger.insights(causal_run, causal_analysis)
display({
    "run_id": causal_run,
    "analysis_id": causal_analysis,
    "episode_count": insights["episode_count"],
    "decision_functions": insights["categories"],
    "reconstruction_stability": insights["reconstruction_stability"],
    "semantica_scoped_insights": insights["semantica_scoped_insights"],
})


Status,Action,Module,Submodule,Progress,ETA,Rate,Time,Extracted
✅,Semantica is building,🧠 kg,CentralityCalculator,100.0%,-,-,0.00s,-
✅,Semantica is building,🧠 kg,CommunityDetector,100.0%,-,-,0.00s,-


🔄 Semantica is building: Calculating degree centrality 🧠 kg CentralityCalculator |░░░░░░░░░░░░░░░| 0.0% ETA: - Rate: - Time: 0.00s Extracted: -

{'run_id': '20260827T105131502195Z_SYN0001_STORE_390_date_of_initial_diagnosis_policy_bundle',
 'analysis_id': 'analysis-96d8ce57c0016c307499',
 'episode_count': 7,
 'decision_functions': {'standing': 3, 'where_to_look': 3, 'which_wins': 1},
 'reconstruction_stability': {'semantics': 'RECONSTRUCTION_STABILITY',
  'mean': 0.5,
  'min': 0.5,
  'max': 0.5},
 'semantica_scoped_insights': {'total_decisions': 7,
  'categories': {'where_to_look': 3, 'standing': 3, 'which_wins': 1},
  'outcomes': {'ACTION:LIST_DOCUMENTS': 1,
   'ACTION:SEARCH': 1,
   'ACTION:READ': 1,
   'MERELY_MENTIONS': 1,
   'CAN_ESTABLISH': 2,
   'SELECT_CANDIDATE:c1': 1},
  'confidence_stats': {'mean': 0.5,
   'min': 0.5,
   'max': 0.5,
   'median': 0.5,
   'semantics': 'RECONSTRUCTION_STABILITY'},
  'advanced_analytics': {'graph_metrics': {'node_count': 10,
    'edge_count': 14,
    'node_types': {'decision': 7, 'category': 3},
    'edge_types': {'belongs_to': 7, 'INFLUENCED': 7}},
   'centrality_analysis': {'centrality

## 11. Open the interactive Explorer view

Use the same selected run and ledger in a terminal:

```bash
acr review-ui <RUN_ID> --run-dir <LEDGER_PARENT>/<RUN_ID> \
  --ledger <LEDGER_PATH> --port 8877
```

Explorer is the delivery shell, while ACR's Decisions workspace supplies the
chart-review narrative, step-by-step audit controls, trace links, and durable human
review provenance. Generic Semantica graph visualization is not the primary review
surface.


In [11]:
command = (
    f"acr review-ui {causal_run} --run-dir {display_path(LEDGER_PATH.parent / causal_run)} "
    f"--ledger {display_path(LEDGER_PATH)} --port 8877"
)
display(Markdown(f"```bash\n{command}\n```"))

closure = {
    "schema": "acr.postdoc_semantica_capabilities.v1",
    "ledger": display_path(LEDGER_PATH),
    "context_graph": stats,
    "claims": {
        "native_similarity_query_executed": True,
        "exact_divergence_found_in_this_cohort": bool(divergence),
        "evidenced_causal_chain_found": bool(hop_rows),
        "policy_direct_binding_queue_found": bool(affected["affected_decisions"]),
        "provenance_chain_valid": integrity["valid"],
        "scoped_insights_executed": insights["episode_count"] > 0,
    },
    "authority_limits": {
        "similarity": "comparison candidates, not clinical precedent",
        "impact": "re-audit candidates, not counterfactual answer change",
        "provenance": "lineage/integrity, not semantic correctness",
        "policy_change": "historical direct bindings, not automatic non-compliance",
    },
}
output = ROOT / "runs/postdoc-notebook-output/03_semantica_capabilities.json"
output.write_text(json.dumps(closure, ensure_ascii=False, indent=2) + "\n")
display(Markdown(
    f"**Notebook 3 closed.** Machine-readable summary: `{display_path(output)}`."
))


```bash
acr review-ui 20260827T105131502195Z_SYN0001_STORE_390_date_of_initial_diagnosis_policy_bundle --run-dir runs/policy-experiment-20260827/20260827T105131502195Z_SYN0001_STORE_390_date_of_initial_diagnosis_policy_bundle --ledger runs/policy-experiment-20260827/experiment-ledger.json --port 8877
```

**Notebook 3 closed.** Machine-readable summary: `runs/postdoc-notebook-output/03_semantica_capabilities.json`.